# Goal

Берём обученного агента (`18n_ppo_tr_frostbite_11:1`) и нахлобучиваем LoRA адаптер, чтобы натренить агента играть лучше на 3-ем уровне. В идеале агент должен проходить без потери жизни.

Используем опыт `50.3a`, чтобы не перебирать разные LoRA параметры. Также **добавляем стохастики (bailey_safe_random_spawn)** в начальные условия, чтобы был сильнее exploration.

# Control plane

In [34]:
TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_11.ipynb'
OPTUNA_DECISIVE_METRIC = 'rollout/episode_lp_conv'

# @launchit.collect
GRID_SEARCH_SPACE = dict(
    random_seed=list(range(1, 101, 10)),
    lora_r_alpha=['16;32', '24;48', '32;64'],
)
# @launchit.stop

if 'GRID_SEARCH_SPACE' in globals():
    import math
    LAUNCHES_COUNT = math.prod(map(len, GRID_SEARCH_SPACE.values()))
    assert LAUNCHES_COUNT > 0
else:
    LAUNCHES_COUNT = 12

# Manual override
# LAUNCHES_COUNT = 10 * 3 - 13

# set_hyperparameters

In [35]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.random_seed = optuna_trial.suggest_categorical('random_seed', GRID_SEARCH_SPACE['random_seed'])
    HP.general.is_torch_deterministic = False # нафиг детерминизм
    HP.general.is_torch_compile = False # НЕ РАБОТАЕТ С LoRA, нужен глубокий рефакторинг
    HP.general.is_torch_amp = True  # без этого памяти будет жраться х1,5 и дольше, поэтому must-have
    HP.general.is_torch_amp_on_rollout = False  # AMP на роллауте тоже нафиг
    
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    wm_parent = '18d_world_model_09:224'
    HP.vision_head.parent = dict(model=wm_parent, weights=wm_parent)
    HP.vision_head.is_trainable = False
    HP.vision_head.is_saving = False

    parent = '18n_ppo_tr_frostbite_11:1'
    HP.encoder.parent = dict(model=wm_parent, weights=parent)
    HP.encoder.is_trainable = False
    HP.encoder.is_saving = False

    HP.agent.parent = dict(weights=parent, lora=None)
    HP.agent.sequence_length = 33
    HP.agent.sequence_ages = [0, 1, 2, 3, 8, 16, 24, 32] # two time scales with steps 1 (~0.13 second) and 8 (~1 second)
    HP.agent.action_plan_length = 10 
    HP.agent.d_model = 256 
    HP.agent.transformer = dict(layers_count=3, heads_count=4, dim_feedforward=1024, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    lora_r_alpha = optuna_trial.suggest_categorical('lora_r_alpha', GRID_SEARCH_SPACE['lora_r_alpha'])
    lora_r, lora_alpha = lora_r_alpha.split(';')
    HP.agent.lora = dict(trainable_adapter='3', r=int(lora_r), alpha=int(lora_alpha), dropout=0.0, bias='none')
    HP.agent.is_trainable = True 
    HP.agent.is_saving = dict(weights=False, lora=True)
    
    # Test params
    HP.test.break_on_level_passed = False
    HP.test.envs_count = 30
    HP.test.env_stories = None # just standard game

    # Rollout params
    HP.rollout.steps_count = 512 # how many steps to run in a single policy rolllout
    l3 = lambda cls: f'com.develorium.neurolab.frostbite_ram:level3:1:cls={cls}'
    patch = 'last_life+no_score+no_igloo+temperature_45+bailey_safe_random_spawn'
    HP.rollout.env_stories = {
        # Main block
        '0:6':   (l3('train1'), patch), 
        '6:12':  (l3('train2'), patch), 
        '12:18': (l3('train3'), patch), 
        '18:24': (l3('train4'), patch), 
        '24:30': (l3('train5'), patch), 
    }
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = 2_000_000 # total number of steps 
    
    HP.ppo.epochs_count = 2 
    HP.ppo.batch_size = 1024
    HP.ppo.learn_rate = f'const(0.00025)'
    HP.ppo.optimizer = 'AdamW'

    HP.ppo.value_loss = dict(coef=0.2)
    HP.ppo.entropy_loss = dict(coef=f'linear(0.05, 0.02)') 
    HP.ppo.consistency_loss = dict(type='MSE', coef=0.1)
    HP.ppo.prediction_loss = dict(coef=0.1)
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization

    # Reporting
    HP.report.rollout_env_groups = None
    HP.report.follow_up_freq = 4

    return HP

# Results

Тут уже получается мясорубка. Из-за доп. стохастичности многие дохнут, но выжившие показывают неплохие результаты. `lora_r_alpha=24;48` выглядит неплохо и смогла принести `:97` экземпляр. Тут сразу видно, что его `levels_passed_mean` пошёл вверх. 

По записи видео `:97` видно, что агент три уровня прошёл на одной жизни! Круто!

А что если довести `episode_lp` реально выше 0.5, насколько вырастет `levels_passed_mean.` Короче, почкуемся от `:97` и вперёд дотренивать адаптер.

|что|`18n_study_50.3b`|`18n_study_50.3a`|
|---|---|---|
|episode_lp|<img src="./img/episode_lp_50.3b.png">|<img src="./img/episode_lp_50.3a.png">|
|levels_passed_mean|<img src="./img/levels_passed_mean_50.3b.png">|<img src="./img/levels_passed_mean_50.3a.png">|

**Выводы**
1) стохастичность вкупе с более `lora_r_alpha=24;48` дала интересный экземпляр
2) смотрим, что будет на дотренирование уже самого адаптера

# System

In [36]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [37]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_11.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_11',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_50.3b/18n_study_50.3b.ipynb',
 'optuna_study_name': '18n_study_50.3b',
 'optuna_study_serial': '50.3b',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_50.3b/18n_study_50.3b.optuna'}

In [38]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [39]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [40]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC=OPTUNA_DECISIVE_METRIC,
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [41]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [42]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while LAUNCHES_COUNT is None or completed_launches_count < LAUNCHES_COUNT:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if LAUNCHES_COUNT is None or (completed_launches_count + len(futures) < LAUNCHES_COUNT):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-11-03 13:15:38,405] Using an existing study with name '18n_study_50.3b' instead of creating a new one.


2026.11.03-13:15:38.842283     0.418 >> Model instance registered, version=102
2026.11.03-13:15:38.861218     0.007 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_11-launch102.ipynb"
2026.11.03-13:15:38.861692     0.007 >> 6.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_11:102"; running launches=1
2026.11.03-13:16:10.074559     0.445 >> Model instance registered, version=103
2026.11.03-13:16:10.088229     0.005 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_11-launch103.ipynb"
2026.11.03-13:16:10.088557     0.005 >> 5.3 idle runners exist, submitted launch "18n_ppo_tr_frostbite_11:103"; running launches=2
2026.11.03-13:16:41.394616     0.612 >> Model instance registered, version=104
2026.11.03-13:16:41.407320     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_11-launch104.ipynb"
2026.11.03-13:16:41.407576     0.005 >> 4.3 idle runners exist, submitted launch "18n_ppo_tr_frostbite_11

In [43]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-11-03 15:52:31,023] Using an existing study with name '18n_study_50.3b' instead of creating a new one.


2026.11.03-15:52:31.025421     5.041 >> Study statistics: 
2026.11.03-15:52:31.032848     0.007 >> 	Number of finished trials: 30
2026.11.03-15:52:31.034141     0.001 >> 	Number of pruned trials: 0
2026.11.03-15:52:31.034746     0.001 >> 	Number of complete trials: 30
2026.11.03-15:52:31.035367     0.001 >> Best trial:
2026.11.03-15:52:31.036211     0.001 >> 	Value: 7.594130570037858
2026.11.03-15:52:31.036655     0.000 >> 	Model version: 97
2026.11.03-15:52:31.036999     0.000 >> 	Params: 
2026.11.03-15:52:31.037290     0.000 >> 		random_seed: 71
2026.11.03-15:52:31.037598     0.000 >> 		lora_r_alpha: 24;48


In [44]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

2026.11.03-15:52:31.046752     0.009 >> Top trials:
2026.11.03-15:52:31.048623     0.002 >> 1) 18n_ppo_tr_frostbite_11:97, value=7.594130570037858
2026.11.03-15:52:31.049183     0.001 >> 2) 18n_ppo_tr_frostbite_11:102, value=5.502121102962874
2026.11.03-15:52:31.049921     0.001 >> 3) 18n_ppo_tr_frostbite_11:109, value=4.131845303542286


In [45]:
values = []

for trial in study.trials:
    if trial.values is not None:
        values.append(trial.values[0])

values = np.array(values)
values.mean(), values.std()

(np.float64(1.6068050182785094), np.float64(1.7690123402304905))